In [2]:
#Regresion Lineal
#Arboles de decision
#random forest
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score

from sklearn.model_selection import train_test_split
import joblib

In [3]:
train_df = pd.read_csv('../Data/train.csv')
test_df = pd.read_csv('../Data/test.csv')
validation_df = pd.read_csv('../Data/validation.csv')

# Separate target column
train_X = train_df.drop(columns=['Depression'])
train_Y = train_df['Depression']

test_X = test_df.drop(columns=['Depression'])
test_Y = test_df['Depression']

validation_X = validation_df.drop(columns=['Depression'])
validation_Y = validation_df['Depression']

Se va a realizar un experimento donde se varian hiperparametros y se busca la mejor solucion

In [4]:
# Hiperparámetros a probar
C_values = [0.1, 1, 10]
kernel_values = ['linear', 'rbf', 'poly']
gamma_values = ['scale', 'auto']

results = []

Se entrena el modelo con todos los posibles valores de hiperparametros y se selecciona el que tenga mejores métricas


In [15]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


# Loop para grid search manual
for C in C_values:
    for kernel in kernel_values:
        for gamma in gamma_values:
            # Inicializar el modelo SVM con los hiperparámetros
            model = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)

            # Entrenar el modelo
            model.fit(train_X, train_Y)

            # Hacer predicciones sobre el conjunto de validación
            pred = model.predict(validation_X)

            # Calcular la matriz de confusión
            cm = confusion_matrix(validation_Y, pred)

            # Calcular métricas
            acc = accuracy_score(validation_Y, pred)
            precision = precision_score(validation_Y, pred, average='weighted')
            recall = recall_score(validation_Y, pred, average='weighted')
            f1 = f1_score(validation_Y, pred, average='weighted')

            # Guardar los resultados en una lista
            results.append({
                'C': C,
                'kernel': kernel,
                'gamma': gamma,
                'Accuracy': acc,
                'Precision': precision,
                'Recall': recall,
                'F1 Score': f1,
                'Confusion Matrix': cm
            })


In [16]:
# Convert the results to a DataFrame
results_df = pd.DataFrame(results)
results_df

,C,kernel,gamma,Accuracy,F1 Score,Precision,Recall,Confusion Matrix
0,0.1,linear,scale,0.842767,0.841514,NaN,NaN,NaN
1,0.1,linear,auto,0.842767,0.841514,NaN,NaN,NaN
2,0.1,rbf,scale,0.616352,0.501800,NaN,NaN,NaN
3,0.1,rbf,auto,0.748428,0.732807,NaN,NaN,NaN
4,0.1,poly,scale,0.647799,0.586138,NaN,NaN,NaN
5,0.1,poly,auto,0.823899,0.822809,NaN,NaN,NaN
6,1.0,linear,scale,0.830189,0.828835,NaN,NaN,NaN
7,1.0,linear,auto,0.830189,0.828835,NaN,NaN,NaN
8,1.0,rbf,scale,0.786164,0.780065,NaN,NaN,NaN
9,1.0,rbf,auto,0.836478,0.835466,NaN,NaN,NaN


In [17]:
# Seleccionar los mejores parámetros con base en F1 Score (o Accuracy)
best_params = results_df.loc[results_df['F1 Score'].idxmax()]
best_params

C                       10.0
kernel                   rbf
gamma                  scale
Accuracy            0.855346
F1 Score            0.853636
Precision                NaN
Recall                   NaN
Confusion Matrix         NaN
Name: 14, dtype: object

Se entrena el modelo con los mejores parámetros.

In [18]:
best_model = SVC(
    C=best_params['C'],
    kernel=best_params['kernel'],
    gamma=best_params['gamma'],
    random_state=42
)

# Entrenar con todo el conjunto de entrenamiento
best_model.fit(train_X, train_Y)

SVC(C=np.float64(10.0), random_state=42)

## Finalmente se usan los datos de "test" para validar las métricas del modelo.

In [19]:
# Evaluate the model on the test set
test_pred = best_model.predict(test_X)
test_mse = mean_squared_error(test_Y, test_pred)
test_r2 = r2_score(test_Y, test_pred)

# Display the final model metrics on the test set
print(f"Test MSE: {test_mse}")
print(f"Test R2: {test_r2}")

Test MSE: 0.16080402010050251
Test R2: 0.3187847667950362


Como ejemplo: se tiene un nuevo inmueble dado por los siguientes datos y se debe predecir el precio.

In [20]:
new_data = {
  "Age": [26.0],
  "Academic Pressure": [4.0],
  "CGPA": [8.78],
  "Study Satisfaction": [2.0],
  "Work/Study Hours": [5.0],
  "Financial Stress": [2.0],
  "Gender_Male": [0.0],
  "Dietary Habits_Moderate": [1.0],
  "Dietary Habits_Unhealthy": [0.0],
  "Degree_B.Arch": [0.0],
  "Degree_B.Com": [1.0],
  "Degree_B.Ed": [0.0],
  "Degree_B.Pharm": [0.0],
  "Degree_B.Tech": [0.0],
  "Degree_BA": [0.0],
  "Degree_BBA": [0.0],
  "Degree_BCA": [0.0],
  "Degree_BE": [0.0],
  "Degree_BHM": [0.0],
  "Degree_BSc": [0.0],
  "Degree_LLB": [0.0],
  "Degree_LLM": [0.0],
  "Degree_M.Com": [0.0],
  "Degree_M.Ed": [0.0],
  "Degree_M.Pharm": [0.0],
  "Degree_M.Tech": [0.0],
  "Degree_MA": [0.0],
  "Degree_MBA": [0.0],
  "Degree_MBBS": [0.0],
  "Degree_MCA": [0.0],
  "Degree_MD": [0.0],
  "Degree_ME": [0.0],
  "Degree_MSc": [0.0],
  "Degree_PhD": [0.0],
  "Have you ever had suicidal thoughts ?_Yes": [1.0],
  "Family History of Mental Illness_Yes": [0.0],
  "City_Ahmedabad": [0.0],
  "City_Bangalore": [0.0],
  "City_Bhopal": [0.0],
  "City_Chennai": [0.0],
  "City_Delhi": [0.0],
  "City_Faridabad": [0.0],
  "City_Ghaziabad": [0.0],
  "City_Hyderabad": [0.0],
  "City_Indore": [0.0],
  "City_Jaipur": [0.0],
  "City_Kalyan": [0.0],
  "City_Kanpur": [0.0],
  "City_Kolkata": [0.0],
  "City_Lucknow": [0.0],
  "City_Ludhiana": [0.0],
  "City_Meerut": [0.0],
  "City_Mumbai": [0.0],
  "City_Nagpur": [0.0],
  "City_Nashik": [0.0],
  "City_Patna": [0.0],
  "City_Pune": [0.0],
  "City_Rajkot": [0.0],
  "City_Srinagar": [0.0],
  "City_Surat": [0.0],
  "City_Thane": [0.0],
  "City_Vadodara": [0.0],
  "City_Varanasi": [0.0],
  "City_Vasai-Virar": [0.0],
  "City_Visakhapatnam": [0.0],
  "Sleep Duration_'7-8 hours'": [1.0],
  "Sleep Duration_'Less than 5 hours'": [0.0],
  "Sleep Duration_'More than 8 hours'": [0.0]
}



# Convertir el nuevo dato en un DataFrame
new_df = pd.DataFrame(new_data)

# Usar el modelo entrenado (best_model) para hacer la predicción
predicted_price = best_model.predict(new_df)

# Mostrar el resultado de la predicción
print(f"Predicted: {predicted_price[0]}")

Predicted: 1


In [21]:
new_data = {
  "Age": [33.0],
  "Academic Pressure": [5.0],
  "CGPA": [9.54],
  "Study Satisfaction": [1.0],
  "Work/Study Hours": [7.0],
  "Financial Stress": [2.0],
  "Gender_Male": [1.0],
  "Dietary Habits_Moderate": [0.0],
  "Dietary Habits_Unhealthy": [0.0],
  "Degree_B.Arch": [0.0],
  "Degree_B.Com": [0.0],
  "Degree_B.Ed": [0.0],
  "Degree_B.Pharm": [1.0],
  "Degree_B.Tech": [0.0],
  "Degree_BA": [0.0],
  "Degree_BBA": [0.0],
  "Degree_BCA": [0.0],
  "Degree_BE": [0.0],
  "Degree_BHM": [0.0],
  "Degree_BSc": [0.0],
  "Degree_LLB": [0.0],
  "Degree_LLM": [0.0],
  "Degree_M.Com": [0.0],
  "Degree_M.Ed": [0.0],
  "Degree_M.Pharm": [0.0],
  "Degree_M.Tech": [0.0],
  "Degree_MA": [0.0],
  "Degree_MBA": [0.0],
  "Degree_MBBS": [0.0],
  "Degree_MCA": [0.0],
  "Degree_MD": [0.0],
  "Degree_ME": [0.0],
  "Degree_MSc": [0.0],
  "Degree_PhD": [0.0],
  "Have you ever had suicidal thoughts ?_Yes": [1.0],
  "Family History of Mental Illness_Yes": [0.0],
  "City_Ahmedabad": [0.0],
  "City_Bangalore": [0.0],
  "City_Bhopal": [0.0],
  "City_Chennai": [0.0],
  "City_Delhi": [0.0],
  "City_Faridabad": [0.0],
  "City_Ghaziabad": [0.0],
  "City_Hyderabad": [0.0],
  "City_Indore": [0.0],
  "City_Jaipur": [0.0],
  "City_Kalyan": [0.0],
  "City_Kanpur": [0.0],
  "City_Kolkata": [0.0],
  "City_Lucknow": [0.0],
  "City_Ludhiana": [0.0],
  "City_Meerut": [0.0],
  "City_Mumbai": [0.0],
  "City_Nagpur": [0.0],
  "City_Nashik": [0.0],
  "City_Patna": [0.0],
  "City_Pune": [0.0],
  "City_Rajkot": [0.0],
  "City_Srinagar": [0.0],
  "City_Surat": [0.0],
  "City_Thane": [0.0],
  "City_Vadodara": [0.0],
  "City_Varanasi": [0.0],
  "City_Vasai-Virar": [0.0],
  "City_Visakhapatnam": [0.0],
  "Sleep Duration_'7-8 hours'": [0.0],
  "Sleep Duration_'Less than 5 hours'": [0.0],
  "Sleep Duration_'More than 8 hours'": [0.0]
}




# Convertir el nuevo dato en un DataFrame
new_df = pd.DataFrame(new_data)

# Usar el modelo entrenado (best_model) para hacer la predicción
predicted_price = best_model.predict(new_df)

# Mostrar el resultado de la predicción
print(f"Predicted: {predicted_price[0]}")

Predicted: 1


En nuestro modelo de clasificación utilizando Support Vector Machine (SVM) con un parámetro C = 10.0, un kernel rbf y gamma = scale, alcanzamos una exactitud (accuracy) del 85.53%, lo que significa que el modelo fue capaz de clasificar correctamente la mayoría de las muestras. El F1 Score obtenido fue de 85.36%, lo cual indica un buen equilibrio entre la precisión y el recall, a pesar de que no contamos con los valores específicos de precisión y recall, ni con la matriz de confusión para hacer un análisis más detallado. Aun así, estos resultados muestran que el modelo tiene un rendimiento competitivo, y nos motivan a completar las métricas faltantes en futuras evaluaciones para una mejor interpretación del comportamiento del clasificador.